In [10]:
import pandas as pd
import joblib
from tensorflow.keras.models import load_model

# Cargar objetos
scaler = joblib.load('scaler.pkl')
columnas_dummies = joblib.load('X_columns.pkl')
modelo = load_model('model.keras')


In [15]:
columnas_dummies

['estu_edad',
 'estu_tipodocumento_CC',
 'estu_tipodocumento_Otro',
 'estu_tipodocumento_TI',
 'cole_area_ubicacion_RURAL',
 'cole_area_ubicacion_URBANO',
 'cole_calendario_A',
 'cole_calendario_B',
 'cole_calendario_OTRO',
 'cole_genero_FEMENINO',
 'cole_genero_MASCULINO',
 'cole_genero_MIXTO',
 'cole_jornada_COMPLETA',
 'cole_jornada_MAÑANA',
 'cole_jornada_NOCHE',
 'cole_jornada_SABATINA',
 'cole_jornada_TARDE',
 'cole_jornada_UNICA',
 'cole_naturaleza_NO OFICIAL',
 'cole_naturaleza_OFICIAL',
 'estu_depto_reside_AMAZONAS',
 'estu_depto_reside_ANTIOQUIA',
 'estu_depto_reside_ARAUCA',
 'estu_depto_reside_ATLANTICO',
 'estu_depto_reside_BOGOTÁ',
 'estu_depto_reside_BOLIVAR',
 'estu_depto_reside_BOYACA',
 'estu_depto_reside_CALDAS',
 'estu_depto_reside_CAQUETA',
 'estu_depto_reside_CASANARE',
 'estu_depto_reside_CAUCA',
 'estu_depto_reside_CESAR',
 'estu_depto_reside_CHOCO',
 'estu_depto_reside_CORDOBA',
 'estu_depto_reside_CUNDINAMARCA',
 'estu_depto_reside_EXTRANJERO',
 'estu_depto_re

In [13]:
def transformar_input(datos_usuario_df):
    # Normalizar
    datos_usuario_df['estu_edad'] = scaler.transform(datos_usuario_df[['estu_edad']])

    # One-hot encoding
    datos_usuario_encoded = pd.get_dummies(datos_usuario_df)
    datos_usuario_encoded = pd.get_dummies(datos_usuario_df).astype(int)

    # Añadir columnas faltantes
    columnas_faltantes = list(set(columnas_dummies) - set(datos_usuario_encoded.columns))
    faltantes_df = pd.DataFrame(0, index=datos_usuario_encoded.index, columns=columnas_faltantes)

    # Concatenar y ordenar columnas
    datos_usuario_encoded = pd.concat([datos_usuario_encoded, faltantes_df], axis=1)
    datos_usuario_encoded = datos_usuario_encoded[columnas_dummies]  # asegurar orden

    return datos_usuario_encoded


In [24]:
nuevo_usuario = pd.DataFrame([{
    'estu_tipodocumento': 'TI',
    'cole_area_ubicacion': 'URBANO',
    'cole_calendario': 'A',
    'cole_genero': 'FEMENINO',
    'cole_jornada': 'Mañana',
    'cole_naturaleza': 'OFICIAL',
    'estu_depto_reside': 'BOGOTA',
    'estu_genero': 'FEMENINO',
    'fami_cuartoshogar': '3',
    'fami_educacionmadre': 'SECUNDARIA',
    'fami_educacionpadre': 'SECUNDARIA',
    'fami_estratovivienda': '5',
    'fami_personashogar': '4',
    'fami_tieneautomovil': 'Si',
    'fami_tienecomputador': 'Si',
    'fami_tieneinternet': 'Si',
    'fami_tienelavadora': 'Si',
    'desemp_ingles': 'B+',
    'estu_edad': 17.0
}])

# Transformar
nuevo_usuario_procesado = transformar_input(nuevo_usuario)

nuevo_usuario_procesado

,estu_edad,estu_tipodocumento_CC,estu_tipodocumento_Otro,estu_tipodocumento_TI,cole_area_ubicacion_RURAL,cole_area_ubicacion_URBANO,cole_calendario_A,cole_calendario_B,cole_calendario_OTRO,cole_genero_FEMENINO,...,fami_tienecomputador_Si,fami_tieneinternet_No,fami_tieneinternet_Si,fami_tienelavadora_No,fami_tienelavadora_Si,desemp_ingles_A-,desemp_ingles_A1,desemp_ingles_A2,desemp_ingles_B+,desemp_ingles_B1
0,0,0,0,1,0,1,1,0,0,1,...,1,0,1,0,1,0,0,0,1,0


In [30]:
# Predecir
prediccion = modelo.predict(nuevo_usuario_procesado)

if prediccion[0][0] > 0.5:
    print(f"El estudiante tiene una alta probabilidad de ser admitido. ({prediccion[0][0]})")
else:
    print(f"El estudiante tiene una baja probabilidad de ser admitido. ({prediccion[0][0]})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
El estudiante tiene una alta probabilidad de ser admitido. (0.8473637104034424)
